[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/00-langchain-intro.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/00-langchain-intro.ipynb)

#### [LangChain Handbook](https://github.com/pinecone-io/examples/tree/master/learn/generation/langchain/handbook)

# Intro to LangChain

LangChain is a popular framework that allow users to quickly build apps and pipelines around **L**arge **L**anguage **M**odels. It can be used for chatbots, RAG, agents, and much more.

The core idea of the library is that we can _"chain"_ together different components to create more advanced use-cases around LLMs. These chains (better thought of as pipelines or workflows) may consist of various components from several modules:

* **Prompt templates**: Prompt templates are, well, templates for different types of prompts. Like "chatbot" style templates, ELI5 question-answering, etc

* **LLMs**: Large language models like GPT-4.1, Claude 4, etc

* **Tool / function calling**: Allow us to augment our LLMs with additional abilities / information sources.

* **Agents**: Agents act as the framework that integrates LLMs and tools.LLMs are packaged into logical loops of operations with tools like web search, **R**etrieval **A**ugmented **G**eneration (RAG), or code execution.

* **Memory**: Short-term memory, long-term memory.

In [1]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU   langchain==0.3.25   langchain-huggingface==0.3.0   langchain-google-genai==2.0.10   sniffio anyio pyparsing

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22

Note: you may need to restart the kernel to use updated packages.


# Using LLMs in LangChain

LangChain supports several LLM providers, like Hugging Face and OpenAI.

Let's start our exploration of LangChain by learning how to use a few of these different LLM integrations.

## Hugging Face

For Hugging Face models we need a Hugging Face Hub API token. We can find this by first getting an account at [HuggingFace.co](https://huggingface.co/) and clicking on our profile in the top-right corner > click *Settings* > click *Access Tokens* > click *New Token* > set *Token type* to `Fine-grained` with the following user or organization permissions:

* **Inference** - Make calls to Inference Providers
* **Inference** - Make calls to your Inference Endpoints
* **Inference** - Manage your Inference Endpoints

After generating the token, enter it below:

In [2]:
import os
from getpass import getpass

token = os.getenv('HF_TOKEN') or \
    getpass("Hugging Face API Token: ")

We can then generate text using a HF Hub model (we'll use `microsoft/Phi-3-mini-4k-instruct`) using the Inference API built into Hugging Face Hub.

_(The default Inference API doesn't use specialized hardware and so can be slow, particularly for larger models)_

In [3]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain import PromptTemplate, LLMChain
import os

# Use HuggingFaceEndpoint with Phi-3-mini-4k-instruct
llm = HuggingFaceEndpoint(
    repo_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    max_new_tokens=100,
    temperature=0.7,
    provider="hf-inference",
    huggingfacehub_api_token=token
)

# Build prompt template
template = """Question: {question}

Answer: """
prompt = PromptTemplate(template=template, input_variables=["question"])

# we chain together the prompt -> LLM with LCEL (more on this later)
llm_chain = prompt | llm

question = "Which NFL team won the Super Bowl in the 2010 season?"

print(llm_chain.invoke(question))

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/hf-inference/models/microsoft/Phi-3-mini-4k-instruct' (Request ID: Root=1-6a3834f1-1df7e6bc33e0f3a06d414f3d;35df9337-0a40-4cea-8fce-b9c96c04e5fa)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

If we'd like to ask multiple questions we can by passing a list of dictionary objects, where the dictionaries must contain the input variable set in our prompt template (`"question"`) that is mapped to the question we'd like to ask.

In [4]:
qs = [
    {'question': "Which NFL team won the Super Bowl in the 2010 season?"},
    {'question': "If I am 6 ft 4 inches, how tall am I in centimeters?"},
    {'question': "Who was the 12th person on the moon?"},
    {'question': "How many eyes does a blade of grass have?"}
]
res = llm_chain.batch(qs)

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/hf-inference/models/microsoft/Phi-3-mini-4k-instruct' (Request ID: Root=1-6a3834f2-5b3587b645d5ec6504dc1ef0;126c1fe6-aa50-4b0f-88d1-d8b54dfa9b1c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [5]:
for question, response in zip(qs, res):
    print("="*100)
    print(f"QUESTION: {question}")
    print(f"RESPONSE: {response}")
    print("="*100 + "\n")

NameError: name 'res' is not defined

## Google Gemini

We can also use Google's Gemini LLMs. The process is similar, we need to
give our API key which can be retrieved from
[Google AI Studio](https://aistudio.google.com/app/apikey). We then pass the API key below:

In [6]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or     getpass("Enter your Google API key: ")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

Gemini models are accessed directly via the Google AI API using the key set above.

Then we decide on which model we'd like to use. We'll go with `gemini-2.5-flash`:

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize with a modern model
gemini_llm = ChatGoogleGenerativeAI(
    temperature=0.7,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# gemini_llm = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0.7,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


The `gemini_llm` object behaves like any other LangChain chat model and can be dropped into the same chains.

We'll use the same simple question-answer prompt template as before with the Hugging Face example. The only change is that we now pass our Gemini LLM `gemini_llm`:

In [8]:
llm_chain = prompt | gemini_llm

print(llm_chain.invoke(question))

content='The Green Bay Packers won Super Bowl XLV (45) in February 2011, which was the championship game for the 2010 NFL season.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--019eeb8e-d2e7-76f2-a455-be8ef392b5bb-0' usage_metadata={'input_tokens': 23, 'output_tokens': 36, 'total_tokens': 235, 'input_token_details': {'cache_read': 0}}


Alternatively we can batch questions as before:

In [9]:
res = llm_chain.batch(qs)

for question, response in zip(qs, res):
    print("="*100)
    print(f"QUESTION: {question}")
    print(f"RESPONSE: {response}")
    print("="*100 + "\n")

QUESTION: {'question': 'Which NFL team won the Super Bowl in the 2010 season?'}
RESPONSE: content='The Green Bay Packers won Super Bowl XLV, which concluded the 2010 NFL season.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--019eeb8e-d9df-7e51-8627-53979fb6ed8a-0' usage_metadata={'input_tokens': 23, 'output_tokens': 21, 'total_tokens': 202, 'input_token_details': {'cache_read': 0}}

QUESTION: {'question': 'If I am 6 ft 4 inches, how tall am I in centimeters?'}
RESPONSE: content="Let's break this down:\n\n1.  **Convert feet to inches:**\n    *   1 foot = 12 inches\n    *   6 feet * 12 inches/foot = 72 inches\n\n2.  **Add the remaining inches:**\n    *   72 inches + 4 inches = 76 inches\n\n3.  **Convert total inches to centimeters:**\n    *   1 inch = 2.54 centimeters\n    *   76 inches * 2.54 cm/inch = **193.04 cm**\n\nSo, you are **193.04 centimeters** tall." additional_kwarg

---